In [8]:
import sys

sys.path.insert(0, '/home/caron/Bureau/ST-GNN-for-wildifre-prediction/Prediction')
sys.path.insert(0, '/home/caron/Bureau/ST-GNN-for-wildifre-prediction/Prediction/GNN')

from tools import *

In [9]:
model = pickle.load(open('firemen/firepoint/2x2/train/occurence_default/check_z-score/full_all_departement_0_None_node/GRUAtt_full_full_10_0_all_one_nbsinister-kmeans-5-Class-Dept_classification_flwki-id{departement}/GRUAtt_full_full_10_0_all_one_nbsinister-kmeans-5-Class-Dept_classification_flwki-id{departement}.pkl', 'rb'))

In [10]:
model.model

GRU(
  (gru): GRU(111, 128, num_layers=2, batch_first=True, dropout=0.03)
  (norm): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.03, inplace=False)
  (context_layer): SpatialContext(
    (dyn_in): Linear(in_features=128, out_features=64, bias=True)
    (stat_in): Linear(in_features=1, out_features=64, bias=True)
    (q_proj): Linear(in_features=64, out_features=64, bias=True)
    (k_proj): Linear(in_features=64, out_features=64, bias=True)
    (v_proj): Linear(in_features=64, out_features=64, bias=True)
    (out_proj): Linear(in_features=64, out_features=64, bias=True)
    (dropout): Dropout(p=0.03, inplace=False)
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (linear1): Linear(in_features=64, out_features=256, bias=True)
  (linear2): Linear(in_features=256, out_features=64, bias=True)
  (output_layer): Linear(in_features=64, out_features=5, bias=True)
  (act_func1): ReLU()
  (act_func2): ReLU()
  (out

In [11]:
small_df = model.df_test[model.df_test['departement'] == 6]

x, y =  model.predict(small_df, return_y=True)

In [12]:
test_loader = model.create_test_loader(model.graph, small_df)

In [13]:
model.model.eval()

with torch.no_grad():

    for _, data in enumerate(test_loader, 0):
        inputs, orilabels_, _ = data
        for H in range(model.horizon + 1):
            orilabels = orilabels_[:, :, -1 - (model.horizon - H)]
            orilabels[:, -1] = orilabels[:,  -1 ] > 0 if model.task_type == 'binary' else orilabels[:,  -1 ]
            inputs_horizon = model.compute_inputs(inputs,  -1 - (model.horizon - H), "current" if H == 0 else "futur")
            
            output, logits, hidden, a = model.model(inputs_horizon, z_prev=None)
    

ValueError: not enough values to unpack (expected 4, got 3)